# 00. Definicja problemu i hipotez

**Etap z planu pracy:** zdefiniowanie problemu i hipotez badawczych.

Ten notebook porzadkuje pytanie badawcze, cel projektu, zakres analizy i sposob operacjonalizacji hipotez na nowo wyczyszczonej bazie `database/NajnowszaWersjaBazy1205_prepared.csv`.


## Pytanie badawcze

Czy na podstawie tekstu, emocji, historii interakcji i prostych cech sieciowych mozna wyjasnic lub przewidziec negatywne odniesienia miedzy subredditami?

Główna zmienna celu to `is_negative_link`, przygotowana z `LINK_SENTIMENT`: `1` oznacza link negatywny, `0` pozostale linki.


## Hipotezy

**H1. Eskalacja emocjonalna**

Prawdopodobienstwo linku negatywnego miedzy dwoma subredditami wzrasta, jesli w poprzednich 24 godzinach wystapily miedzy nimi interakcje o wysokim natezeniu slownictwa zwiazanego z gniewem (`LIWC_Anger`).

**H2. Zlozonosc poznawcza**

Teksty towarzyszace negatywnym linkom maja nizsza zlozonosc jezykowa niz teksty w linkach pozytywnych.

**H3. Model multimodalny**

Modele ML powinny przewidywac negatywne odniesienia z F1-score powyzej 75%, laczac cechy lingwistyczne, emocjonalne, transformerowe i strukturalne.


## Operacjonalizacja

- H1: testujemy `high_previous_anger_24h` oraz statystyki `prev_pair_*_24h` wzgledem `is_negative_link`.
- H2: porownujemy wybrane cechy zlozonosci z `prepared_dataset_metadata.json`.
- H3: trenujemy modele na `split_chronological`, z metrykami skupionymi na klasie negatywnej.
- Komponent Hugging Face traktujemy w dwoch warstwach: jako gotowe kolumny `Content_Sentiment` / `Content_Score` oraz jako opcjonalna komorke z nowszym modelem transformerowym, wylaczona domyslnie ze wzgledu na pobieranie modelu i czas wykonania.


In [1]:
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "database").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_PATH = PROJECT_ROOT / "database" / "NajnowszaWersjaBazy1205_prepared.csv"
RAW_DATA_PATH = PROJECT_ROOT / "database" / "NajnowszaWersjaBazy1205.csv"
METADATA_PATH = PROJECT_ROOT / "outputs" / "prepared_dataset_metadata.json"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "czysta_baza"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)
pd.set_option("display.float_format", lambda value: f"{value:.4f}")

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_PATH:", DATA_PATH)
print("Prepared data exists:", DATA_PATH.exists())
print("Output dir:", OUTPUT_DIR)

assert DATA_PATH.exists(), f"Brakuje pliku z przygotowana baza: {DATA_PATH}"


PROJECT_ROOT: C:\Users\szymon\projekt_reddit
DATA_PATH: C:\Users\szymon\projekt_reddit\database\NajnowszaWersjaBazy1205_prepared.csv
Prepared data exists: True
Output dir: C:\Users\szymon\projekt_reddit\outputs\czysta_baza


In [2]:
with open(METADATA_PATH, "r", encoding="utf-8") as file:
    metadata = json.load(file)

df_preview = pd.read_csv(DATA_PATH, nrows=5)
print("Liczba kolumn w prepared CSV:", len(df_preview.columns))
print("Target:", metadata["target"])
print("Split:", metadata["split_column"])
display(df_preview[[
    "POST_ID",
    "TIMESTAMP",
    "SOURCE_SUBREDDIT",
    "TARGET_SUBREDDIT",
    "LINK_SENTIMENT",
    "is_negative_link",
    "Content_Sentiment",
    "high_previous_anger_24h",
]])


Liczba kolumn w prepared CSV: 106
Target: is_negative_link
Split: split_chronological


,POST_ID,TIMESTAMP,SOURCE_SUBREDDIT,TARGET_SUBREDDIT,LINK_SENTIMENT,is_negative_link,Content_Sentiment,high_previous_anger_24h
0,1u4nrps,2013-12-31 16:39:58,leagueoflegends,teamredditteams,1,0,0,False
1,1u4sjvs,2013-12-31 17:37:55,nfl,cfb,1,0,0,False
2,1u4qkd,2013-12-31 18:18:37,theredlion,soccer,-1,1,1,False
3,1u4w7bs,2013-12-31 18:35:44,dogemarket,dogecoin,1,0,0,False
4,1u5df2s,2013-12-31 22:27:50,gfycat,india,1,0,0,False


## Rezultat etapu

Hipotezy zostaly przepisane na mierzalne zmienne w oczyszczonej bazie. Kolejne notebooki ida etapami z `plan_pracy.md`: opis danych, analiza przygotowania, przeglad podejsc, implementacja metod, interpretacja i szkic prezentacji.
